# CMU 10-799 HW1 Part II Runner

Use this notebook to run the Part II DDPM training and sampling workflow on Google Colab.

The source of truth for code is the repository, not this notebook. Edit code locally, commit/push, then pull the latest code here before training.


## 0. GPU Runtime

In Colab, use `Runtime -> Change runtime type -> GPU`, then run these checks.


In [7]:
!nvidia-smi

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


Fri May  1 11:29:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive

Colab local disk disappears when the runtime resets. Save long training outputs to Drive.


In [8]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/cmu-10799-diffusion"
!mkdir -p "{DRIVE_ROOT}/logs" "{DRIVE_ROOT}/checkpoints" "{DRIVE_ROOT}/samples"
print(DRIVE_ROOT)


ValueError: Mountpoint must not already contain files

## 2. Clone Or Update Repo

If this is a fresh runtime, clone your fork. If it already exists, pull the latest code.


In [9]:
%cd /content

import os

repo_dir = "/content/cmu-10799-diffusion"
repo_url = "https://github.com/Eryc123Y/cmu-10799-diffusion.git"  # Change if needed.

if os.path.isdir(repo_dir):
    %cd /content/cmu-10799-diffusion
    !git pull origin main
else:
    !git clone "{repo_url}"
    %cd /content/cmu-10799-diffusion

!git status --short --branch
!git log -1 --oneline


/content
/content/cmu-10799-diffusion
From https://github.com/Eryc123Y/cmu-10799-diffusion
 * branch            main       -> FETCH_HEAD
Already up to date.
## main...origin/main
c047a2b (HEAD -> main, origin/main, origin/HEAD) Implement DDPM model, training, and sampling workflow; add Colab notebook for easy execution


## 3. Install Dependencies

Colab usually already has CUDA PyTorch. This cell installs the extra packages used by the starter code.


In [10]:
!pip install -q einops PyYAML tqdm scipy pandas matplotlib datasets huggingface-hub torch-fidelity wandb

import yaml
import einops
import datasets
import torch_fidelity

print("imports ok")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.6 MB/s eta 0:00:00
imports ok


## 4. Create A Colab Config

This is a practical starter config. If memory is tight, reduce `batch_size` first.


In [11]:
%%writefile configs/ddpm_colab.yaml
data:
  dataset: "celeba"
  root: "./data/celeba-subset"
  from_hub: true
  repo_name: "electronickale/cmu-10799-celeba64-subset"
  image_size: 64
  channels: 3
  num_workers: 2
  pin_memory: true
  augment: true

model:
  base_channels: 64
  channel_mult: [1, 2, 2, 4]
  num_res_blocks: 2
  attention_resolutions: [16]
  num_heads: 4
  dropout: 0.1
  use_scale_shift_norm: true

training:
  batch_size: 64
  learning_rate: 0.0002
  weight_decay: 0.0
  betas: [0.9, 0.999]
  ema_decay: 0.9999
  ema_start: 1000
  gradient_clip_norm: 1.0
  num_iterations: 100000
  log_every: 100
  sample_every: 2000
  save_every: 5000
  num_samples: 16

ddpm:
  num_timesteps: 1000
  beta_start: 0.0001
  beta_end: 0.02

sampling:
  num_steps: 1000
  sampler: "ddpm"

infrastructure:
  seed: 42
  device: "cuda"
  num_gpus: 1
  mixed_precision: true
  compile_model: false

checkpoint:
  dir: "./checkpoints"
  resume: null

logging:
  dir: "/content/drive/MyDrive/cmu-10799-diffusion/logs"
  wandb:
    enabled: false
    project: "cmu-10799-diffusion"
    entity: null


Writing configs/ddpm_colab.yaml


## 5. Syntax And Tiny Tensor Smoke Test

Run this before any long training. It checks importability, model output shape, DDPM loss, and a tiny sampling loop.


In [17]:
!python -m compileall -q src train.py sample.py

import torch
from src.models.unet import UNet
from src.methods.ddpm import DDPM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    in_channels=3,
    out_channels=3,
    base_channels=64,
    channel_mult=(1, 2, 2, 4),
    num_res_blocks=1,
    attention_resolutions=[16],
    num_heads=4,
    dropout=0.1,
    use_scale_shift_norm=True,
    image_size=64,
).to(device)

method = DDPM(
    model=model,
    device=device,
    num_timesteps=10,
    beta_start=1e-4,
    beta_end=2e-2,
)

x_0 = torch.randn(2, 3, 64, 64, device=device)
t = torch.randint(0, method.num_timesteps, (2,), device=device)

x_t, noise = method.forward_process(x_0, t)
loss, metrics = method.compute_loss(x_0)
samples = method.sample(batch_size=2, image_shape=(3, 64, 64), num_steps=10)

print("x_t:", x_t.shape)
print("noise:", noise.shape)
print("loss:", loss.shape, metrics)
print("samples:", samples.shape)


x_t: torch.Size([2, 3, 64, 64])
noise: torch.Size([2, 3, 64, 64])
loss: torch.Size([]) {'loss': 1.1087944507598877, 'mse': 1.1087944507598877}
samples: torch.Size([2, 3, 64, 64])


## 6. Optional: Download Dataset To Local Disk

The config uses `from_hub: true`, so this may not be necessary. If loading from hub is slow, download/cache the dataset locally.


In [18]:
# Optional. Uncomment if you want a local cache.
# !python download_dataset.py --output_dir ./data/celeba-subset --split all


## 7. Single-Batch Sanity Check

Run this before full training. If this fails to lower loss or crashes during sampling, full training will waste GPU time.


In [7]:
# This should be much cheaper than full training.
# Stop manually once the behavior is clear.
!python train.py --method ddpm --config configs/ddpm_colab.yaml --overfit-single-batch


DEVICE CONFIGURATION
✓ Single GPU training
  - Device: cuda (NVIDIA A100-SXM4-40GB)
  - Config device setting: cuda
  - Mixed precision: True
Creating data loader...
Attempt to use cached dataset from: ./data/celeba-subset
⬇ Downloading dataset from HuggingFace Hub: electronickale/cmu-10799-celeba64-subset
  (This may take a few minutes on first run)
Using HuggingFace cache directory: ./data/celeba-subset
README.md: 2.36kB [00:00, 4.47MB/s]
data/train-00000-of-00001.parquet: 100% 471M/471M [00:02<00:00, 194MB/s] 
Generating train split: 100% 63715/63715 [00:01<00:00, 54014.94 examples/s]
Loaded 63715 images from train split
Dataset size: 63715
Batches per epoch: 995
Creating model...
Model parameters: 15,605,635 (15.61M)
Creating ddpm...
Logging to: /content/drive/MyDrive/cmu-10799-diffusion/logs/ddpm_20260430_220209

Starting training from step 0...
Total iterations: 100000
DEBUG MODE: Overfitting to a single batch
--------------------------------------------------
Cached single batch

## 8. Full Training

Record the GPU type, start/end time, final iteration, loss curve, sample grids, and checkpoint path for the report.


In [8]:
!python train.py --method ddpm --config configs/ddpm_colab.yaml


DEVICE CONFIGURATION
✓ Single GPU training
  - Device: cuda (NVIDIA A100-SXM4-40GB)
  - Config device setting: cuda
  - Mixed precision: True
Creating data loader...
Attempt to use cached dataset from: ./data/celeba-subset
⬇ Downloading dataset from HuggingFace Hub: electronickale/cmu-10799-celeba64-subset
  (This may take a few minutes on first run)
Using HuggingFace cache directory: ./data/celeba-subset
Loaded 63715 images from train split
Dataset size: 63715
Batches per epoch: 995
Creating model...
Model parameters: 15,605,635 (15.61M)
Creating ddpm...
Logging to: /content/drive/MyDrive/cmu-10799-diffusion/logs/ddpm_20260430_220531

Starting training from step 0...
Total iterations: 100000
--------------------------------------------------
  2% 1999/100000 [02:46<2:24:33, 11.30it/s, loss=0.0222, steps/s=11.85]
Generating samples at step 2000...
  4% 3998/100000 [05:45<2:23:44, 11.13it/s, loss=0.0191, steps/s=11.85] 
Generating samples at step 4000...
  5% 4998/100000 [07:23<2:08:29,

: 

## 9. Generate A 16-Sample Grid

Update `CHECKPOINT_PATH` to the checkpoint you want to evaluate.


In [19]:
CHECKPOINT_PATH = "TODO_REPLACE_WITH_CHECKPOINT.pt"

!python sample.py \
  --checkpoint "{CHECKPOINT_PATH}" \
  --method ddpm \
  --num_samples 16 \
  --batch_size 16 \
  --grid \
  --output /content/drive/MyDrive/cmu-10799-diffusion/samples/part_ii_16_grid.png


Using device: cuda
Loading checkpoint from TODO_REPLACE_WITH_CHECKPOINT.pt...
Traceback (most recent call last):
  File "/content/cmu-10799-diffusion/sample.py", line 207, in <module>
    main()
  File "/content/cmu-10799-diffusion/sample.py", line 127, in main
    model, config, ema = load_checkpoint(args.checkpoint, device)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cmu-10799-diffusion/sample.py", line 44, in load_checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.

## 10. KID Evaluation With 1k Samples

Run these cells in order from a fresh Colab web runtime. They are self-contained: they mount Drive, prepare the repo, locate `ddpm_final.pt` from Google Drive, generate 1000 samples, export 1000 real images, and compute KID. This version does not use a browser upload widget; the checkpoint should already be synced to Google Drive.


In [21]:
# KID setup for Colab web: mount Drive, prepare repo, and locate ddpm_final.pt.
# This block does NOT open an upload widget. Put ddpm_final.pt in Google Drive first.
from pathlib import Path
import shutil

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"
DRIVE_PROJECT = MYDRIVE / "cmu-10799-diffusion"
DRIVE_HOMEWORK = DRIVE_PROJECT / "HW1_CMU_10799_Spring_2026"
DRIVE_CODE_ZIP = DRIVE_HOMEWORK / "hw1_code.zip"
DRIVE_CHECKPOINT = DRIVE_HOMEWORK / "ddpm_final.pt"

LOCAL_REPO = Path("/content/cmu-10799-diffusion")
LOCAL_CHECKPOINT = LOCAL_REPO / "ddpm_final.pt"
REPO_URL = "https://github.com/Eryc123Y/cmu-10799-diffusion.git"

# Mount Drive robustly. If Colab already has MyDrive, do not remount.
if MYDRIVE.exists():
    print("Google Drive already mounted:", MYDRIVE)
else:
    try:
        drive.mount(str(DRIVE_ROOT))
    except ValueError as err:
        if "Mountpoint must not already contain files" not in str(err):
            raise
        print("/content/drive is non-empty; retrying with force_remount=True")
        drive.mount(str(DRIVE_ROOT), force_remount=True)

# Show the exact expected files. This makes account/sync mistakes obvious.
print("Expected Drive homework folder:", DRIVE_HOMEWORK)
print("Expected checkpoint:", DRIVE_CHECKPOINT)
print("Expected code zip:", DRIVE_CODE_ZIP)

# Prepare the repo. Prefer the exported code zip because it matches the trained checkpoint.
if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
LOCAL_REPO.mkdir(parents=True, exist_ok=True)

if DRIVE_CODE_ZIP.exists():
    print("Using code zip from Drive:", DRIVE_CODE_ZIP)
    !unzip -q "{DRIVE_CODE_ZIP}" -d "{LOCAL_REPO}"
else:
    print("Code zip not found in Drive; cloning GitHub repo instead:", REPO_URL)
    shutil.rmtree(LOCAL_REPO)
    !git clone "{REPO_URL}" "{LOCAL_REPO}"

%cd /content/cmu-10799-diffusion

# Install packages needed by sampling, dataset loading, and KID evaluation.
!pip install -q einops PyYAML tqdm scipy pandas matplotlib datasets huggingface-hub torch-fidelity wandb

# Locate the checkpoint. Prefer the exact expected path, then search MyDrive.
checkpoint_candidates = []
if DRIVE_CHECKPOINT.exists():
    checkpoint_candidates.append(DRIVE_CHECKPOINT)
checkpoint_candidates.extend(sorted(MYDRIVE.glob("**/ddpm_final.pt")))
checkpoint_candidates.extend(sorted(MYDRIVE.glob("**/ddpm_0100000.pt")))
checkpoint_candidates.extend(sorted(MYDRIVE.glob("**/ddpm_0095000.pt")))

# Deduplicate while preserving order.
unique_candidates = []
seen = set()
for candidate in checkpoint_candidates:
    candidate = Path(candidate)
    if str(candidate) in seen:
        continue
    seen.add(str(candidate))
    if candidate.exists():
        unique_candidates.append(candidate)

if not unique_candidates:
    print("No checkpoint found in this Colab-mounted Google Drive account.")
    print("Run this in a notebook cell to inspect visible files:")
    print('!find /content/drive/MyDrive -type f -name "ddpm_final.pt"')
    raise FileNotFoundError(
        "ddpm_final.pt is not visible to Colab. Make sure it has finished syncing to the same Google account used by Colab."
    )

print("Found checkpoint candidates:")
for candidate in unique_candidates:
    print(" -", candidate)
FINAL_CHECKPOINT = str(unique_candidates[0])

# Use /content for KID images because it is faster than writing thousands of files to Drive.
KID_ROOT = Path("/content/kid_eval")
GENERATED_DIR = str(KID_ROOT / "generated_1k")
REAL_DIR = str(KID_ROOT / "real_1k")

print("Repo:", Path.cwd())
print("Final checkpoint:", FINAL_CHECKPOINT)
print("Generated sample dir:", GENERATED_DIR)
print("Real image dir:", REAL_DIR)
!ls -lh sample.py train.py configs/ddpm_colab.yaml "{FINAL_CHECKPOINT}"


Google Drive already mounted.
/content/cmu-10799-diffusion
No checkpoint found under /content/drive/MyDrive yet.
If you just copied ddpm_final.pt into Google Drive on your Mac, wait for sync, then rerun this cell.
Expected Drive path:
/content/drive/MyDrive/cmu-10799-diffusion/HW1_CMU_10799_Spring_2026/ddpm_final.pt
As a fallback, upload this local file when prompted:
/Users/eric/courses/cmu-10799-diffusion/HW1_CMU_10799_Spring_2026/ddpm_final.pt


KeyboardInterrupt: 

In [20]:
# Generate 1000 EMA samples from the trained DDPM.
# This is the slowest KID step because DDPM uses the full 1000-step reverse chain.
!rm -rf "{GENERATED_DIR}"
!mkdir -p "{GENERATED_DIR}"

!python sample.py \
  --checkpoint "{FINAL_CHECKPOINT}" \
  --method ddpm \
  --num_samples 1000 \
  --batch_size 64 \
  --output_dir "{GENERATED_DIR}"

!find "{GENERATED_DIR}" -type f -name "*.png" | wc -l


Using device: cuda
Loading checkpoint from /content/drive/MyDrive/cmu-10799-diffusion/logs/ddpm_20260501_004133/checkpoints/ddpm_final.pt...
Traceback (most recent call last):
  File "/content/cmu-10799-diffusion/sample.py", line 207, in <module>
    main()
  File "/content/cmu-10799-diffusion/sample.py", line 127, in main
    model, config, ema = load_checkpoint(args.checkpoint, device)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cmu-10799-diffusion/sample.py", line 44, in load_checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^

In [ ]:
# Export 1000 real CelebA subset images for torch-fidelity.
# Augmentation is disabled so the reference set is deterministic.
from pathlib import Path
import shutil

import torch
from torchvision.utils import save_image as torchvision_save_image

from train import load_config
from src.data import create_dataloader_from_config, unnormalize

config = load_config("configs/ddpm_colab.yaml")
config["data"]["augment"] = False
config["training"]["batch_size"] = 64

real_dir = Path(REAL_DIR)
if real_dir.exists():
    shutil.rmtree(real_dir)
real_dir.mkdir(parents=True, exist_ok=True)

loader = create_dataloader_from_config(config, split="train")
image_count = 0
for batch in loader:
    images = unnormalize(batch).clamp(0.0, 1.0)
    for image in images:
        torchvision_save_image(image, real_dir / f"{image_count:06d}.png")
        image_count += 1
        if image_count >= 1000:
            break
    if image_count >= 1000:
        break

print(f"Saved {image_count} real images to {real_dir}")


In [ ]:
# Compute KID. Copy these two printed values into Q4c:
# - kernel_inception_distance_mean
# - kernel_inception_distance_std
import torch
from torch_fidelity import calculate_metrics

metrics = calculate_metrics(
    input1=GENERATED_DIR,
    input2=REAL_DIR,
    cuda=torch.cuda.is_available(),
    isc=False,
    fid=False,
    kid=True,
    kid_subset_size=100,
    kid_subsets=10,
    verbose=True,
)

print(metrics)
print("KID mean:", metrics["kernel_inception_distance_mean"])
print("KID std:", metrics["kernel_inception_distance_std"])


## 11. Evidence To Copy Into The Report

- Model size: `15,605,635` parameters, printed by `train.py` at startup.
- Batch size: `training.batch_size = 64` in `configs/ddpm_colab.yaml`.
- Total training iterations: `100000`.
- Training loss trace: `HW1_CMU_10799_Spring_2026/figures/q4a_loss_curve.png`.
- 16-sample grid: `HW1_CMU_10799_Spring_2026/figures/q4b_samples_0100000.png`.
- Final checkpoint: `ddpm_final.pt`. Do not include this large file in the code zip.
- KID: copy the printed `kernel_inception_distance_mean` and `kernel_inception_distance_std` into Q4c after running the KID cells.
